In [ ]:
# 15_effect_size_tost.ipynb
# A-6: effect size (Cohen d), 95% CI, Welch test and TOST equivalence test for
# DiCE vs SHAP guide quality (primary judge). Reads results/llm/judge_scores.csv.
# Runs after notebook 11. Writes results/tables/table_a6_effect_size_tost.csv.
from pathlib import Path
import numpy as np, pandas as pd
from scipy import stats
ROOT = Path.cwd(); ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
TAB = ROOT/"results"/"tables"
js = pd.read_csv(ROOT/"results"/"llm"/"judge_scores.csv")
pj = js[js.judge=="primary"]
def stat(a,b):
    a=np.asarray(a,float); b=np.asarray(b,float); df=len(a)+len(b)-2
    d=a.mean()-b.mean(); se=np.sqrt(a.var(ddof=1)/len(a)+b.var(ddof=1)/len(b))
    sp=np.sqrt(((len(a)-1)*a.var(ddof=1)+(len(b)-1)*b.var(ddof=1))/df)
    t=stats.t.ppf(0.975,df); tt,p=stats.ttest_ind(a,b,equal_var=False)
    tost={m:max(1-stats.t.cdf((d-(-m))/se,df), stats.t.cdf((d-m)/se,df)) for m in (0.5,0.3)}
    return d,d-t*se,d+t*se,d/sp,p,tost
rows=[]
for dim in ["D3_actionability","D7_overall_quality"]:
    a=pj[pj.method=="dice"][dim].dropna().values; b=pj[pj.method=="shap"][dim].dropna().values
    d,lo,hi,dd,p,tost=stat(a,b)
    rows.append([dim,round(a.mean(),3),round(b.mean(),3),round(d,3),round(lo,3),round(hi,3),round(dd,3),round(p,4),round(tost[0.5],4),round(tost[0.3],4)])
out=pd.DataFrame(rows,columns=["Dimension","DiCE_mean","SHAP_mean","Mean_diff","CI_low","CI_high","Cohen_d","Welch_p","TOST_0.5_p","TOST_0.3_p"])
out.to_csv(TAB/"table_a6_effect_size_tost.csv",index=False); print(out.to_string(index=False))
